**A brief introduction of the client:** 

MediCore Health has acquired City Care Clinics **(50 locations)**. The client requires a Unified Patient Data  Lakehouse to merge legacy CSV extracts into the corporate Delta Lake. The source data is **volatile (schema drift), dirty (data quality issues), and highly sensitive (PII**). The target  system must support complex historical tracking (SCD Type 2) for billing while maintaining high  performance for clinical lookups. 


In [0]:
### Setting up the environment
from pyspark.sql import functions as f
from pyspark.sql.types import*
from pyspark import pipelines as dp 
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("MediCore Health").getOrCreate()

**Nature of the Data:** 

The data is received in **CSV file batches at intervals of approximately 5–10 minutes**. However, the arrival pattern is not fixed or predictable. Therefore, an event-based trigger will be configured to ensure that the job is executed only when a new CSV batch file arrives.

**Technical Environment & Constraints&#xA0;**

• **Platform:&#x20;**&#x44;atabricks Serverless Compute. 

• **Storage:&#x20;**&#x55;nity Catalog Volumes (/Volumes/...) or DBFS. 

• **Ingestion Source:&#x20;**&#x4D;anual upload of messy CSVs to the Volume. 

• **Orchestration:&#x20;**&#x54;he solution must be contained in a single Notebook that can be run end-to-end  (idempotent).

In [0]:
### Building the ingestion code with the following requiements specified by the client.
@dp.table(name="resillient_bronze_ingestion",comment="Here we are ingesting our data based on the requirements specified in the above mentioned pointers")

def resillient_bronze_ingestion():
    df=spark.readStream.format("cloudFiles").option("cloudFiles.format","csv").option("cloudFiles.schemaLocation","/Volumes/medicore_health_catalog/staging/raw_data_schema_location").option("inferSchema","false").option("header","true").option("cloudFiles.schemaEvolutionMode","addNewColumns").option("cloudFiles.rescuedDataColumn","_rescued_data").load("/Volumes/medicore_health_catalog/staging/raw_data/").withColumn("ingestion_timestamp",f.current_timestamp()).withColumn("file_name",input_file_name())

    df= df.withColumn("HospitalID",f.col("HospitalID").try_cast("integer"))
    df=df.withColumn("LastVisitDate",f.col("LastVisitDate").try_cast("date"))

    return df

In [0]:
# Retrieve the secret from Databricks Secret Scope and raising an error if the salt key is missing
salt_key = dbutils.secrets.get(scope="pii-secrets",key="ssn-salt-key")
if not salt_key:
    raise ValueError("SSN salt key is missing.") 

### Defining the leak-protection function

def leak_protection(df):
    req_columns=[col for col in df.columns if "ssn" in col.lower()]
    if req_columns:
        raise ValueError(f"The SSN has bene leaked" 
                         f"The leaked ssn details are {req_columns}") 
    return df    

##### Defining the silver table
 
@dp.table(name="silver_transactions_table",comment="After staging the data in the bronze layer now the required transformations have to be applied on the data in adherence to the requirements specified",table_properties={"delta.enableChangeDataFeed": "true"}) 

def silver_transactions_table():
    df1= spark.readStream.table("resillient_bronze_ingestion")
    good_records= df1.filter((f.col("HospitalID").isNotNull())) ### Here we are specifying the criteria of PK not being null
    good_records=good_records.withColumn("SSN",f.sha2(f.concat(f.col("SSN"),f.lit(salt_key)),256))
    good_records= good_records.select(f.col("HospitalID").astype("int"),f.col("Name").astype("string"),f.col("SSN").astype("string"),f.col("Address").astype("string"),f.col("Phone").astype("string"),f.col("Age").astype("int"),f.col("LastVisitDate").astype("date"),f.col("ZipCode").astype("string"))

    good_records=leak_detection(good_records)
    return good_records

@dp.view(name="analytical_view", comment="Here we have to stimulate and create a view that explicitly excludes the Hashed_SSN columns for General Analyticsusers.")

def analytical_view():
    df = spark.read.table("silver_transactions_table")

    restricted_columns = ["SSN"]

    allowed_columns = [column for column in df.columnsif column not in restricted_columns]

    return df.select(*allowed_columns)